# Lock Waits, Deadlocks and the Throughput Cliff

Companion notebook for the [Two Phase Locking and the Lock Manager](https://codemia.io/courses/database_internals/dbi_two_phase_locking_and_the_lock_manager) lesson on Codemia.

**What you will build**: a lock manager with a wait queue, a waits for graph and a deadlock
detector, driven by concurrent transactions that hold their locks until commit. By the end you will
be able to say why adding connections stops helping and then starts hurting, why deadlocks rise with
the fourth power of transaction length, and why sorting the keys before touching them removes the
problem entirely.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(17)

RECORDS = 1000     # rows that can be locked
LOCKS = 8          # locks a transaction takes before it commits
TICKS = 40000      # simulated time steps per run

## Concept recap

Two phase locking has two rules. A transaction takes every lock it needs before it releases any, and
under the strict version that most engines implement it releases them all at commit. That is what
makes the schedule serializable, and it is also what makes a lock a resource held for the whole
remaining life of the transaction rather than for the instant it is used.

Two consequences follow.

The first is waiting. A transaction blocked on a lock holds everything it has already taken, so the
transaction behind it waits too. As concurrency rises, the chance that any particular transaction is
waiting rises faster than linearly, and throughput stops improving long before the hardware is busy.
Past a certain point it falls, because each new transaction mostly adds something else to wait for.

The second is deadlock. Two transactions that take the same two locks in opposite orders will
eventually interleave in the one way that leaves each waiting for the other. No amount of waiting
resolves it, so the engine detects the cycle and kills one of them. The rate at which this happens
rises very steeply with how many locks a transaction takes, which is the single most useful fact in
this notebook.

## A lock manager with a wait queue and a detector

In [ ]:
class Sim:
    """Time stepped two phase locking. One lock acquisition per transaction per tick.

    Transactions hold every lock until they commit, blocked transactions queue on the
    holder, and a cycle in the waits for graph is a deadlock that costs one abort.
    """

    def __init__(self, mpl, locks=LOCKS, records=RECORDS, ordered=False, skew=0.0, seed=17):
        self.gen = np.random.default_rng(seed)
        self.mpl = mpl
        self.locks = locks
        self.records = records
        self.ordered = ordered          # acquire in key order rather than in arrival order
        self.skew = skew                # fraction of accesses aimed at a tenth of the rows
        self.owner = {}                 # record -> transaction holding it
        self.queue = {}                 # record -> transactions waiting, in order
        self.txns = {}
        self.next_id = 0
        self.tick = 0
        self.commits = 0
        self.deadlocks = 0
        self.blocked_ticks = 0
        self.latency = []
        for _ in range(mpl):
            self.start()

    def keys_for(self):
        hot = max(1, self.records // 10)
        pick = []
        while len(pick) < self.locks:
            if self.gen.random() < self.skew:
                k = int(self.gen.integers(0, hot))
            else:
                k = int(self.gen.integers(0, self.records))
            if k not in pick:
                pick.append(k)
        return sorted(pick) if self.ordered else pick

    def start(self, tid=None):
        tid = self.next_id if tid is None else tid
        if tid == self.next_id:
            self.next_id += 1
        self.txns[tid] = {"keys": self.keys_for(), "at": 0, "held": [],
                          "blocked_on": None, "began": self.tick}
        return tid

    def release_all(self, tid):
        t = self.txns[tid]
        for k in t["held"]:
            del self.owner[k]
            waiters = self.queue.get(k)
            if waiters:
                nxt = waiters.pop(0)
                self.owner[k] = nxt
                self.txns[nxt]["held"].append(k)
                self.txns[nxt]["at"] += 1
                self.txns[nxt]["blocked_on"] = None
        t["held"] = []

    def step_txn(self, tid):
        t = self.txns[tid]
        if t["blocked_on"] is not None:
            return
        if t["at"] >= self.locks:
            self.commits += 1
            self.latency.append(self.tick - t["began"])
            self.release_all(tid)
            self.start(tid)
            return
        k = t["keys"][t["at"]]
        holder = self.owner.get(k)
        if holder is None:
            self.owner[k] = tid
            t["held"].append(k)
            t["at"] += 1
        else:
            self.queue.setdefault(k, []).append(tid)
            t["blocked_on"] = k

    def waits_for(self, tid):
        k = self.txns[tid]["blocked_on"]
        return None if k is None else self.owner.get(k)

    def find_cycle(self):
        for tid in list(self.txns):
            seen, cur = [], tid
            for _ in range(self.mpl + 1):
                nxt = self.waits_for(cur)
                if nxt is None:
                    break
                seen.append(cur)
                if nxt == tid:
                    return seen
                cur = nxt
        return None

    def resolve(self, cycle):
        victim = max(cycle, key=lambda t: self.txns[t]["began"])
        t = self.txns[victim]
        if t["blocked_on"] is not None:
            waiters = self.queue.get(t["blocked_on"], [])
            if victim in waiters:
                waiters.remove(victim)
        self.release_all(victim)
        self.deadlocks += 1
        self.start(victim)

    def run(self, ticks=TICKS):
        for self.tick in range(ticks):
            for tid in list(self.txns):
                self.step_txn(tid)
            cycle = self.find_cycle()
            if cycle:
                self.resolve(cycle)
            self.blocked_ticks += sum(
                1 for t in self.txns.values() if t["blocked_on"] is not None)
        return {
            "mpl": self.mpl,
            "throughput": self.commits / ticks,
            "deadlocks_per_1000": 1000 * self.deadlocks / max(1, self.commits),
            "latency": float(np.mean(self.latency)) if self.latency else float("nan"),
            "blocked": self.blocked_ticks / (ticks * self.mpl),
        }

## Throughput against concurrency

In [ ]:
mpls = [1, 2, 4, 8, 16, 32, 64, 128]
curve = [Sim(m, skew=0.5).run() for m in mpls]

print("%6s %12s %12s %12s %10s" %
      ("mpl", "commits/tick", "latency", "deadlocks/1k", "blocked"))
for r in curve:
    print("%6d %12.3f %12.1f %12.1f %9.0f%%" %
          (r["mpl"], r["throughput"], r["latency"], r["deadlocks_per_1000"], 100 * r["blocked"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mpls, [r["throughput"] for r in curve], "o-", color="#5b8def", lw=2)
axes[0].set_xscale("log", base=2)
axes[0].set_xlabel("concurrent transactions")
axes[0].set_ylabel("commits per tick")
axes[0].set_title("throughput")
axes[1].plot(mpls, [100 * r["blocked"] for r in curve], "o-", color="#d1495b", lw=2,
             label="blocked")
axes[1].plot(mpls, [r["deadlocks_per_1000"] for r in curve], "o-", color="#8a5bd6", lw=2,
             label="deadlocks per 1000 commits")
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("concurrent transactions")
axes[1].set_title("what the extra concurrency is doing")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Throughput rises almost linearly while contention is low, peaks at sixteen concurrent transactions
and then falls, until at a hundred and twenty eight it is below what eight were doing. The right
hand panel says why: the time each transaction spends sitting in a wait queue climbs toward all of
it, so each additional transaction mostly adds one more thing for the others to wait behind, and
the deadlock rate adds abandoned work on top of that.

Latency is the number that gets attention first in practice, because it rises before throughput
falls. The run at sixteen is already taking twice as long per transaction as the run at one, and it
is still the fastest configuration here.

The remedy is the unglamorous one: fewer concurrent transactions. A pool sized at the peak of this
curve delivers five times the work of one sized eight times larger, which is the opposite of what a
queue of waiting application threads suggests.

## Deadlocks against transaction length

In [ ]:
print("%8s %14s %16s %18s" %
      ("locks", "deadlocks/1k", "commits/tick", "fourth power model"))
base = None
rows = []
for k in (2, 4, 6, 8, 12, 16):
    r = Sim(32, locks=k, skew=0.5).run()
    rows.append((k, r))
    if k == 8:
        base = r["deadlocks_per_1000"]
for k, r in rows:
    model = base * (k / 8.0) ** 4 if base else float("nan")
    print("%8d %14.1f %16.3f %18.1f" % (k, r["deadlocks_per_1000"], r["throughput"], model))

The classical analysis says the deadlock rate grows with the fourth power of the number of locks a
transaction holds, and the final column scales the measured rate at eight locks by that factor so
the shape can be compared. The two columns are the same order of magnitude at six, twelve and
sixteen locks and diverge badly at two and four, which is what should be expected: the derivation
assumes uniform access and no queueing, and this run has skew and a deep wait queue.

What survives is the steepness. Going from four locks to eight multiplied the measured rate by
forty six, and from eight to sixteen by nine, and the throughput column fell by a factor of sixty
across the same range. The exponent is approximate. The practical reading is not: transaction length
is the most expensive variable in this notebook.

Which turns the usual advice into something quantitative. Splitting a transaction that touches
sixteen rows into two that touch eight each is not a modest improvement, and keeping a transaction
open across a network call is not a modest risk.

## Taking the locks in a fixed order

In [ ]:
print("%-26s %12s %14s %10s" % ("acquisition order", "commits/tick", "deadlocks/1k", "latency"))
for ordered in (False, True):
    r = Sim(32, skew=0.5, ordered=ordered).run()
    print("%-26s %12.3f %14.1f %10.1f" %
          ("sorted by key" if ordered else "as the code touches them",
           r["throughput"], r["deadlocks_per_1000"], r["latency"]))

Sorting the keys before acquiring them removes deadlock completely, and the argument is short enough
to keep: if every transaction takes locks in increasing key order, then a transaction waiting on key
k holds only keys below k, so a cycle would need a transaction waiting on something it already holds.
There is no cycle to find.

Waiting does not go away, because the contention was never caused by the ordering. Transactions
still queue on the same hot rows. What goes away is the aborts, and with them the work those
transactions had already done, which is why throughput rose by seventy per cent and mean latency
fell by a third from one change that costs nothing at runtime.

In application code this is the rule that says update accounts in account number order, and it is
worth stating plainly: a deadlock is not a concurrency problem to be tuned away. It is an ordering
problem, and ordering is free.

## Variations: how much the skew matters

In [ ]:
print("%14s %14s %14s %10s" % ("hot fraction", "commits/tick", "deadlocks/1k", "blocked"))
for skew in (0.0, 0.25, 0.5, 0.75, 0.95):
    r = Sim(32, skew=skew).run()
    print("%14.2f %14.3f %14.1f %9.0f%%" %
          (skew, r["throughput"], r["deadlocks_per_1000"], 100 * r["blocked"]))

The hot fraction is how often a transaction aims at the busiest tenth of the rows. Everything in
this notebook gets worse as it rises, and it rises in real systems for banal reasons: a counter row,
a status table, a queue table where everyone takes the oldest entry.

That is the shape of most lock contention incidents. The system is not short of capacity and the
transactions are not long. A handful of rows are being contended by everything at once, and the fix
is almost always to stop writing to that row rather than to tune the lock manager.

## What to try next

- Add shared locks for readers and let several transactions hold the same key, which is what a real
  lock manager does, and see how much of the waiting disappears for a read heavy workload.
- Implement wait or die and wound or wait, the timestamp schemes that prevent deadlock instead of
  detecting it, and compare aborted work against detection latency.
- Give each transaction a think time between acquisitions, which is what a network round trip is, and
  watch the throughput peak move sharply to the left.
- Measure how long a cycle exists before the detector finds it when detection runs every hundred
  ticks rather than every tick, which is what a real engine's detection interval setting controls.